# 8.1 章节介绍：性能优化进阶

[第7章](../07_performance_basics/README.md)给了通用的优化路径：均衡性 → CV 流水与切块 → 核内流水。本章把镜头拉近，**按执行引擎逐个击破**——同一个 FA kernel 里，Cube、Vector、Scalar 三类单元的瓶颈形态完全不同，优化手段也完全不同。

## 三引擎的瓶颈画像

| 引擎 | 在 FA 里干的活 | 典型瓶颈 | 本章小节 |
|--|--|--|--|
| **Cube** | mm1（Q·Kᵀ）、mm2（P·V）两个矩阵乘 | K 轴装载与搬运效率、L0 容量约束下的切分 | [8.2](08.02_cube_k_split.ipynb) / [8.3](08.03_cube_best_k.ipynb) |
| **Vector** | softmax、mask、量化反量化、update | VF 指令编排、ND↔NZ 转换开销 | [8.4](08.04_vector_vf_opt.ipynb) / [8.5](08.05_vector_dn_modeling.ipynb) |
| **Scalar** | 循环控制、地址计算、同步标志 | 标量循环拖慢整条流水、分支抖动 | [8.6](08.06_scalar_opt.ipynb) |

## Cube：K 切分是核心变量

FA 里两个 mm 的 K 轴长度天差地别：

- **mm1 的 K = D**（head 维），典型 128，往往一次就能全载；
- **mm2 的 K = S2**（KV 序列长），推理场景可达上万，**必须切**。

K 怎么切，直接决定 MTE1 搬运的整块率和 L0 的占用形态。8.2 讲清楚「为什么切、切了对带宽有什么影响」，8.3 给出「拿到一个 shape 怎么算出最优 K 切分」的可动手流程。

## Vector：VF 优化与 DN 选型

- **VF 优化**：第7章 7.6 学了 RegBase 编程的骨架，本章进入指令级——vLoop 怎么切、指令怎么省、尾块怎么兜；
- **DN**：一条「让数据绕过 Vector 直进 Cube」的通路。它不是万能药——8.5 给出 DN vs ND 的建模与选型判据；
- **VF 建模**：一条 VF 指令吃 64 个 FP32，耗时不靠测，靠数指令就能估出来。

## Scalar：最容易被忽视的一环

Scalar 指令虽然「不干活」，但**每一条都在占用发射周期**。标量循环展开、地址增量常量化、分支消除——这些手段单看每条收益都不大，叠在一起却能让 Vector 的等待窗口肉眼可见地缩短。

## 本章学习路线

| 小节 | 主题 | 回答的问题 |
|--|--|--|
| [8.2 Cube 性能（上）：K 切分原理](08.02_cube_k_split.ipynb) | K 轴的角色、L0 容量约束、切分对带宽的影响 | K 为什么必须切？切了会怎样？ |
| [8.3 Cube 性能（下）：最优 K 切分](08.03_cube_best_k.ipynb) | 容量方程、分形对齐、切分方案建模与选优 | 拿到 shape，K 切多大最优？ |
| [8.4 Vector 性能（上）：VF 优化](08.04_vector_vf_opt.ipynb) | vLoop 切分、指令精简、尾块 MaskReg | VF 代码还能怎么快？ |
| [8.5 Vector 性能（下）：DN 与 VF 建模](08.05_vector_dn_modeling.ipynb) | DN 通路、DN vs ND 选型、VF 耗时建模 | 什么时候绕过 Vector 反而更快？ |
| [8.6 Scalar 优化](08.06_scalar_opt.ipynb) | 标量上提、循环展开、分支消除 | 不干活的指令怎么少占周期？ |
| [8.7 章节测验](08.07_chapter_test.ipynb) | 综合测验 | 检验本章掌握程度 |

## 前置知识

- [第7章 性能优化基础](../07_performance_basics/README.md)：Roofline 建模、CV 流水、RegBase 编程骨架
- [第1章 昇腾开发基础](../01_ascend_dev_basics/README.md)：Cube/Vector/Scalar 指令体系、MTE 通道
- [第5章 非量化 FA 开发](../05_flash_attn_non_quantized/README.md)：FA kernel 的四拍循环结构
